# Đánh giá ba mô hình baseline trên tập Test

Ba baseline E01 Decision Tree, E05 Random Forest và E10 MLP được giữ nguyên cấu hình trong notebook nguồn. Mỗi mô hình được huấn luyện trên Train + Validation, sau đó dự đoán cùng tập Test độc lập. Không dùng Test để chọn hoặc điều chỉnh mô hình.

HTML đi kèm được xuất trực tiếp từ notebook đã chạy. Excel gồm điểm tổng thể, kết quả theo lớp và confusion matrix trên Test.

In [1]:
from pathlib import Path
from time import perf_counter
from warnings import catch_warnings, simplefilter
import inspect

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, Side
from openpyxl.utils import get_column_letter
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

PROJECT_ROOT = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project")
RESULT_DIR = PROJECT_ROOT / "result" / "baseline" / "test baseline"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROJECT_ROOT / "data" / "splitted data"
TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"
EXCEL_PATH = RESULT_DIR / "Baseline_Test_Evaluation.xlsx"
TARGET = "Attack_type"
RANDOM_STATE = 42

for required_path in (TRAIN_PATH, VALIDATION_PATH, TEST_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)
print("Data source:", DATA_DIR)
print("Output folder:", RESULT_DIR)

Data source: C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\data\splitted data
Output folder: C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\baseline\test baseline


In [2]:
# Prepare the fixed development partition. Test is not used in fitting or model choices.
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VALIDATION_PATH)
test = pd.read_csv(TEST_PATH)
if any(TARGET not in frame.columns for frame in (train, validation, test)):
    raise ValueError(f"Missing target column {TARGET!r}.")
if not train.columns.equals(validation.columns) or not train.columns.equals(test.columns):
    raise ValueError("Train, Validation, and Test schemas do not match.")
if train.empty or validation.empty or test.empty:
    raise ValueError("A data split is empty.")
if train.isna().any().any() or validation.isna().any().any() or test.isna().any().any():
    raise ValueError("Missing values found in the baseline data partitions.")

X_development = pd.concat(
    [train.drop(columns=[TARGET]), validation.drop(columns=[TARGET])],
    ignore_index=True,
)
y_development = pd.concat([train[TARGET].astype(str), validation[TARGET].astype(str)], ignore_index=True)
X_test = test.drop(columns=[TARGET]).copy()
y_test = test[TARGET].astype(str).copy()
if not X_development.columns.equals(X_test.columns):
    raise ValueError("Test feature columns do not match Train + Validation.")
CLASS_LABELS = sorted(y_development.unique().tolist())
if set(y_test.unique()) != set(CLASS_LABELS):
    raise ValueError("Test and development partitions do not have the same class labels.")

print(f"Train: {len(train):,} rows. Validation: {len(validation):,} rows.")
print(f"Final fit: {len(X_development):,} rows. Test: {len(X_test):,} rows.")
print(f"Features: {X_development.shape[1]}. Classes: {len(CLASS_LABELS)}.")
print("Class distribution in Test:")
display(y_test.value_counts().rename_axis("Class").to_frame("Test support"))

# Verify the saved split audit reports no duplicate feature hashes across partitions.
audit_path = PROJECT_ROOT / "result" / "preprocessing" / "preprocessing_basic" / "step2_summary.json"
if audit_path.is_file():
    import json
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    overlaps = audit.get("cross_split_duplicate_feature_hashes", {})
    if any(overlaps.get(key, 0) != 0 for key in ("train_test", "validation_test")):
        raise ValueError(f"The saved split audit reports overlap with Test: {overlaps}")
    print("Split audit: zero cross-split duplicate feature hashes with Test.")

Train: 82,536 rows. Validation: 17,686 rows.
Final fit: 100,222 rows. Test: 17,687 rows.
Features: 82. Classes: 12.
Class distribution in Test:


,Test support
Class,
DOS_SYN_Hping,13514
Thing_Speak,1148
ARP_poisioning,1143
MQTT_Publish,622
NMAP_UDP_SCAN,388
NMAP_XMAS_TREE_SCAN,301
NMAP_OS_DETECTION,300
NMAP_TCP_scan,150
DDOS_Slowloris,79


Split audit: zero cross-split duplicate feature hashes with Test.


In [3]:
# Common test evaluation. Each pipeline below matches its baseline notebook configuration.
results = {}

def evaluate_baseline(name, pipeline, estimator_step):
    with catch_warnings(record=True) as captured_warnings:
        simplefilter("always")
        started = perf_counter()
        pipeline.fit(X_development, y_development)
        fit_seconds = perf_counter() - started
        y_pred = pipeline.predict(X_test)

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro-F1": f1_score(y_test, y_pred, labels=CLASS_LABELS, average="macro", zero_division=0),
        "Macro-Recall": recall_score(y_test, y_pred, labels=CLASS_LABELS, average="macro", zero_division=0),
        "Macro-Precision": precision_score(y_test, y_pred, labels=CLASS_LABELS, average="macro", zero_division=0),
        "Weighted-F1": f1_score(y_test, y_pred, labels=CLASS_LABELS, average="weighted", zero_division=0),
    }
    report = classification_report(y_test, y_pred, labels=CLASS_LABELS, output_dict=True, zero_division=0)
    per_class = pd.DataFrame([
        {"Class": label, "Precision": report[label]["precision"], "Recall": report[label]["recall"],
         "F1": report[label]["f1-score"], "Support": int(report[label]["support"])}
        for label in CLASS_LABELS
    ])
    matrix = confusion_matrix(y_test, y_pred, labels=CLASS_LABELS)
    estimator = pipeline.named_steps[estimator_step]
    result = {
        "name": name,
        "pipeline": pipeline,
        "metrics": metrics,
        "per_class": per_class,
        "confusion": matrix,
        "predictions": y_pred,
        "fit_seconds": fit_seconds,
        "n_iter": getattr(estimator, "n_iter_", None),
        "warning_count": len(captured_warnings),
        "report": report,
    }
    results[name] = result
    print(f"{name}: fit completed in {fit_seconds:.2f} seconds.")
    if result["n_iter"] is not None:
        print(f"MLP epochs completed: {result['n_iter']}.")
    print(f"Captured training warnings: {result['warning_count']}.")
    display(pd.DataFrame([metrics], index=[name]).T.rename(columns={0: "Test score"}).style.format("{:.6f}"))
    display(per_class.style.format({"Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}"}))
    display(pd.DataFrame(matrix, index=CLASS_LABELS, columns=CLASS_LABELS).rename_axis("Actual").rename_axis("Predicted", axis=1))

    lowest_f1 = per_class.sort_values(["F1", "Support"]).iloc[0]
    lowest_recall = per_class.sort_values(["Recall", "Support"]).iloc[0]
    correct = int((y_test.to_numpy() == np.asarray(y_pred).astype(str)).sum())
    display(Markdown(
        f"Nhận xét: Trên Test, {name} dự đoán đúng {correct:,}/{len(y_test):,} mẫu. "
        f"Macro-F1 là {metrics['Macro-F1']:.4f}, Macro-Recall là {metrics['Macro-Recall']:.4f}. "
        f"Lớp có F1 thấp nhất là {lowest_f1['Class']} (F1 {lowest_f1['F1']:.3f}, "
        f"{int(lowest_f1['Support'])} mẫu). Lớp có Recall thấp nhất là {lowest_recall['Class']} "
        f"(Recall {lowest_recall['Recall']:.3f}, {int(lowest_recall['Support'])} mẫu)."
    ))
    return result

print("Evaluation helper ready. Each fixed baseline will be fitted and tested in separate cells.")

Evaluation helper ready. Each fixed baseline will be fitted and tested in separate cells.


## E01 — Decision Tree baseline

In [4]:
# E01 baseline uses median imputation for numeric features and one-hot encoding for categories.
numeric_e01 = X_development.select_dtypes(include=[np.number]).columns.tolist()
categorical_e01 = [column for column in X_development.columns if column not in numeric_e01]

numeric_pipeline_e01 = Pipeline([
    ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
])
categorical_pipeline_e01 = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__", keep_empty_features=True)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])
preprocessor_e01 = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline_e01, numeric_e01),
        ("categorical", categorical_pipeline_e01, categorical_e01),
    ],
    remainder="drop",
)
e01 = Pipeline([
    ("preprocess", preprocessor_e01),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])
print("Configuration: DecisionTreeClassifier(random_state=42), unrestricted depth.")
result_e01 = evaluate_baseline("E01 Decision Tree", e01, "model")

Configuration: DecisionTreeClassifier(random_state=42), unrestricted depth.


E01 Decision Tree: fit completed in 5.09 seconds.
Captured training warnings: 0.


,E01 Decision Tree
Accuracy,0.998304
Macro-F1,0.973849
Macro-Recall,0.976320
Macro-Precision,0.975128
Weighted-F1,0.998304


,Class,Precision,Recall,F1,Support
0,ARP_poisioning,0.9878,0.9930,0.9904,1143
1,DDOS_Slowloris,0.9872,0.9747,0.9809,79
2,DOS_SYN_Hping,1.0000,1.0000,1.0000,13514
3,MQTT_Publish,0.9984,1.0000,0.9992,622
4,Metasploit_Brute_Force_SSH,1.0000,0.8000,0.8889,5
5,NMAP_FIN_SCAN,0.8000,1.0000,0.8889,4
6,NMAP_OS_DETECTION,1.0000,1.0000,1.0000,300
7,NMAP_TCP_scan,1.0000,1.0000,1.0000,150
8,NMAP_UDP_SCAN,0.9948,0.9948,0.9948,388
9,NMAP_XMAS_TREE_SCAN,1.0000,0.9967,0.9983,301


Predicted,ARP_poisioning,DDOS_Slowloris,DOS_SYN_Hping,MQTT_Publish,Metasploit_Brute_Force_SSH,NMAP_FIN_SCAN,NMAP_OS_DETECTION,NMAP_TCP_scan,NMAP_UDP_SCAN,NMAP_XMAS_TREE_SCAN,Thing_Speak,Wipro_bulb
Actual,,,,,,,,,,,,
ARP_poisioning,1135,0,0,1,0,0,0,0,0,0,7,0
DDOS_Slowloris,1,77,0,0,0,0,0,0,0,0,1,0
DOS_SYN_Hping,0,0,13514,0,0,0,0,0,0,0,0,0
MQTT_Publish,0,0,0,622,0,0,0,0,0,0,0,0
Metasploit_Brute_Force_SSH,1,0,0,0,4,0,0,0,0,0,0,0
NMAP_FIN_SCAN,0,0,0,0,0,4,0,0,0,0,0,0
NMAP_OS_DETECTION,0,0,0,0,0,0,300,0,0,0,0,0
NMAP_TCP_scan,0,0,0,0,0,0,0,150,0,0,0,0
NMAP_UDP_SCAN,1,0,0,0,0,1,0,0,386,0,0,0


Nhận xét: Trên Test, E01 Decision Tree dự đoán đúng 17,657/17,687 mẫu. Macro-F1 là 0.9738, Macro-Recall là 0.9763. Lớp có F1 thấp nhất là NMAP_FIN_SCAN (F1 0.889, 4 mẫu). Lớp có Recall thấp nhất là Metasploit_Brute_Force_SSH (Recall 0.800, 5 mẫu).

## E05 — Random Forest baseline

In [5]:
categorical_e05 = X_development.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_e05 = [column for column in X_development.columns if column not in categorical_e05]
preprocessor_e05 = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_e05),
        ("numeric", "passthrough", numeric_e05),
    ],
    remainder="drop",
)
e05 = Pipeline([
    ("preprocessor", preprocessor_e05),
    ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
])
print("Configuration: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).")
result_e05 = evaluate_baseline("E05 Random Forest", e05, "model")

Configuration: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).


E05 Random Forest: fit completed in 6.14 seconds.
Captured training warnings: 0.


,E05 Random Forest
Accuracy,0.998643
Macro-F1,0.974462
Macro-Recall,0.960651
Macro-Precision,0.996673
Weighted-F1,0.998628


,Class,Precision,Recall,F1,Support
0,ARP_poisioning,0.9870,0.9965,0.9917,1143
1,DDOS_Slowloris,0.9868,0.9494,0.9677,79
2,DOS_SYN_Hping,1.0000,1.0000,1.0000,13514
3,MQTT_Publish,1.0000,1.0000,1.0000,622
4,Metasploit_Brute_Force_SSH,1.0000,0.6000,0.7500,5
5,NMAP_FIN_SCAN,1.0000,1.0000,1.0000,4
6,NMAP_OS_DETECTION,1.0000,1.0000,1.0000,300
7,NMAP_TCP_scan,1.0000,1.0000,1.0000,150
8,NMAP_UDP_SCAN,0.9897,0.9948,0.9923,388
9,NMAP_XMAS_TREE_SCAN,1.0000,0.9967,0.9983,301


Predicted,ARP_poisioning,DDOS_Slowloris,DOS_SYN_Hping,MQTT_Publish,Metasploit_Brute_Force_SSH,NMAP_FIN_SCAN,NMAP_OS_DETECTION,NMAP_TCP_scan,NMAP_UDP_SCAN,NMAP_XMAS_TREE_SCAN,Thing_Speak,Wipro_bulb
Actual,,,,,,,,,,,,
ARP_poisioning,1139,0,0,0,0,0,0,0,0,0,4,0
DDOS_Slowloris,1,75,0,0,0,0,0,0,3,0,0,0
DOS_SYN_Hping,0,0,13514,0,0,0,0,0,0,0,0,0
MQTT_Publish,0,0,0,622,0,0,0,0,0,0,0,0
Metasploit_Brute_Force_SSH,2,0,0,0,3,0,0,0,0,0,0,0
NMAP_FIN_SCAN,0,0,0,0,0,4,0,0,0,0,0,0
NMAP_OS_DETECTION,0,0,0,0,0,0,300,0,0,0,0,0
NMAP_TCP_scan,0,0,0,0,0,0,0,150,0,0,0,0
NMAP_UDP_SCAN,2,0,0,0,0,0,0,0,386,0,0,0


Nhận xét: Trên Test, E05 Random Forest dự đoán đúng 17,663/17,687 mẫu. Macro-F1 là 0.9745, Macro-Recall là 0.9607. Lớp có F1 thấp nhất là Metasploit_Brute_Force_SSH (F1 0.750, 5 mẫu). Lớp có Recall thấp nhất là Metasploit_Brute_Force_SSH (Recall 0.600, 5 mẫu).

## E10 — MLP baseline

In [6]:
categorical_e10 = ["proto", "service"]
missing_e10 = [column for column in categorical_e10 if column not in X_development.columns]
if missing_e10:
    raise ValueError(f"Missing MLP categorical features: {missing_e10}")
numeric_e10 = X_development.select_dtypes(include=np.number).columns.tolist()
if set(numeric_e10 + categorical_e10) != set(X_development.columns):
    raise ValueError("An MLP feature is neither numeric nor categorical.")

ohe_params = {"handle_unknown": "ignore"}
if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
    ohe_params["sparse_output"] = False
else:
    ohe_params["sparse"] = False
preprocessor_e10 = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_e10),
        ("categorical", OneHotEncoder(**ohe_params), categorical_e10),
    ],
    remainder="drop",
    sparse_threshold=0,
)
e10 = Pipeline([
    ("preprocess", preprocessor_e10),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        learning_rate_init=0.001,
        batch_size=512,
        max_iter=60,
        n_iter_no_change=8,
        early_stopping=False,
        random_state=RANDOM_STATE,
        verbose=False,
    )),
])
print("Configuration: MLP (64,), StandardScaler, batch_size=512, max_iter=60, alpha=0.0001.")
result_e10 = evaluate_baseline("E10 MLP", e10, "classifier")

Configuration: MLP (64,), StandardScaler, batch_size=512, max_iter=60, alpha=0.0001.


E10 MLP: fit completed in 40.93 seconds.
MLP epochs completed: 60.
Captured training warnings: 1.


,E10 MLP
Accuracy,0.995194
Macro-F1,0.956872
Macro-Recall,0.938252
Macro-Precision,0.985548
Weighted-F1,0.995133


,Class,Precision,Recall,F1,Support
0,ARP_poisioning,0.9804,0.9641,0.9722,1143
1,DDOS_Slowloris,0.9688,0.7848,0.8671,79
2,DOS_SYN_Hping,1.0000,1.0000,1.0000,13514
3,MQTT_Publish,0.9984,1.0000,0.9992,622
4,Metasploit_Brute_Force_SSH,1.0000,0.6000,0.7500,5
5,NMAP_FIN_SCAN,1.0000,1.0000,1.0000,4
6,NMAP_OS_DETECTION,1.0000,1.0000,1.0000,300
7,NMAP_TCP_scan,0.9868,1.0000,0.9934,150
8,NMAP_UDP_SCAN,0.9576,0.9897,0.9734,388
9,NMAP_XMAS_TREE_SCAN,1.0000,0.9967,0.9983,301


Predicted,ARP_poisioning,DDOS_Slowloris,DOS_SYN_Hping,MQTT_Publish,Metasploit_Brute_Force_SSH,NMAP_FIN_SCAN,NMAP_OS_DETECTION,NMAP_TCP_scan,NMAP_UDP_SCAN,NMAP_XMAS_TREE_SCAN,Thing_Speak,Wipro_bulb
Actual,,,,,,,,,,,,
ARP_poisioning,1102,0,0,1,0,0,0,2,0,0,38,0
DDOS_Slowloris,1,62,0,0,0,0,0,0,16,0,0,0
DOS_SYN_Hping,0,0,13514,0,0,0,0,0,0,0,0,0
MQTT_Publish,0,0,0,622,0,0,0,0,0,0,0,0
Metasploit_Brute_Force_SSH,2,0,0,0,3,0,0,0,0,0,0,0
NMAP_FIN_SCAN,0,0,0,0,0,4,0,0,0,0,0,0
NMAP_OS_DETECTION,0,0,0,0,0,0,300,0,0,0,0,0
NMAP_TCP_scan,0,0,0,0,0,0,0,150,0,0,0,0
NMAP_UDP_SCAN,3,0,0,0,0,0,0,0,384,0,1,0


Nhận xét: Trên Test, E10 MLP dự đoán đúng 17,602/17,687 mẫu. Macro-F1 là 0.9569, Macro-Recall là 0.9383. Lớp có F1 thấp nhất là Metasploit_Brute_Force_SSH (F1 0.750, 5 mẫu). Lớp có Recall thấp nhất là Metasploit_Brute_Force_SSH (Recall 0.600, 5 mẫu).

## So sánh kết quả Test

In [7]:
summary = pd.DataFrame([
    {"Model": name, **result["metrics"], "Fit seconds": result["fit_seconds"],
     "MLP epochs": result["n_iter"] if result["n_iter"] is not None else np.nan}
    for name, result in results.items()
])
summary["Test samples"] = len(y_test)
display(summary.style.format({column: "{:.6f}" for column in summary.select_dtypes(include="number").columns}))

# Save Test-only metrics, per-class details, and confusion matrices.
workbook = Workbook()
summary_ws = workbook.active
summary_ws.title = "Test Summary"
for row in [summary.columns.tolist()] + summary.values.tolist():
    summary_ws.append([value.item() if isinstance(value, np.generic) else value for value in row])

per_class_ws = workbook.create_sheet("Per-Class Test")
per_class_ws.append(["Model", "Class", "Precision", "Recall", "F1", "Support"])
for name, result in results.items():
    for row in result["per_class"].itertuples(index=False, name=None):
        per_class_ws.append([name, *row])

for sheet_name, key in [("E01 Confusion", "E01 Decision Tree"),
                        ("E05 Confusion", "E05 Random Forest"),
                        ("E10 Confusion", "E10 MLP")]:
    ws = workbook.create_sheet(sheet_name)
    ws.append(["Actual \\ Predicted"] + CLASS_LABELS)
    for label, row in zip(CLASS_LABELS, results[key]["confusion"]):
        ws.append([label] + [int(value) for value in row])

line = Side(style="thin", color="D9D9D9")
for ws in workbook.worksheets:
    for row in ws.iter_rows():
        for cell in row:
            cell.font = Font(name="Calibri", size=10, bold=(cell.row == 1), color="000000")
            cell.border = Border(left=line, right=line, top=line, bottom=line)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
    ws.sheet_view.showGridLines = True
    ws.freeze_panes = None
for column, width in {"A": 26, "B": 17, "C": 17, "D": 17, "E": 17, "F": 17, "G": 17, "H": 17, "I": 17}.items():
    summary_ws.column_dimensions[column].width = width
per_class_ws.column_dimensions["A"].width = 25
per_class_ws.column_dimensions["B"].width = 32
for column in range(3, 7):
    per_class_ws.column_dimensions[get_column_letter(column)].width = 16
for ws in workbook.worksheets[2:]:
    ws.column_dimensions["A"].width = 32
    for column in range(2, len(CLASS_LABELS) + 2):
        ws.column_dimensions[get_column_letter(column)].width = 22
workbook.save(EXCEL_PATH)
print("Saved Excel test results:", EXCEL_PATH)
print("The Test set was not used for model selection or tuning.")

,Model,Accuracy,Macro-F1,Macro-Recall,Macro-Precision,Weighted-F1,Fit seconds,MLP epochs,Test samples
0,E01 Decision Tree,0.998304,0.973849,0.976320,0.975128,0.998304,5.090231,nan,17687.000000
1,E05 Random Forest,0.998643,0.974462,0.960651,0.996673,0.998628,6.143213,nan,17687.000000
2,E10 MLP,0.995194,0.956872,0.938252,0.985548,0.995133,40.928933,60.000000,17687.000000


Saved Excel test results: C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\baseline\test baseline\Baseline_Test_Evaluation.xlsx
The Test set was not used for model selection or tuning.
